# 01 — Exploratory Data Analysis
**Project:** Tennis Era Dominance  
**Goal:** Understand the shape of our data before building Elo or running tests.

Questions we want to answer here:
1. How are Grand Slam wins distributed across players and decades?
2. Which eras look 'dominant' visually?
3. What does our ATP match data cover — completeness, surface split, date range?

In [1]:
import pandas as pd

matches = pd.read_csv('../data/processed/atp_matches_clean.csv')
slams = pd.read_csv('../data/processed/slams_clean.csv')

print('Match columns:', matches.columns.tolist())
print('Slam columns:', slams.columns.tolist())

Match columns: ['tourney_id', 'tourney_name', 'surface', 'draw_size', 'tourney_level', 'tourney_date', 'match_num', 'winner_id', 'winner_seed', 'winner_entry', 'winner_name', 'winner_hand', 'winner_ht', 'winner_ioc', 'winner_age', 'loser_id', 'loser_seed', 'loser_entry', 'loser_name', 'loser_hand', 'loser_ht', 'loser_ioc', 'loser_age', 'score', 'best_of', 'round', 'minutes', 'w_ace', 'w_df', 'w_svpt', 'w_1stIn', 'w_1stWon', 'w_2ndWon', 'w_SvGms', 'w_bpSaved', 'w_bpFaced', 'l_ace', 'l_df', 'l_svpt', 'l_1stIn', 'l_1stWon', 'l_2ndWon', 'l_SvGms', 'l_bpSaved', 'l_bpFaced', 'winner_rank', 'winner_rank_points', 'loser_rank', 'loser_rank_points']
Slam columns: ['YEAR', 'TOURNAMENT', 'WINNER', 'RUNNER-UP', 'WINNER_NATIONALITY', 'WINNER_ATP_RANKING', 'RUNNER-UP_ATP_RANKING', 'WINNER_LEFT_OR_RIGHT_HANDED', 'TOURNAMENT_SURFACE', 'WINNER_PRIZE']


In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Style
sns.set_theme(style='whitegrid', palette='deep')
plt.rcParams['figure.figsize'] = (12, 5)

# ----- LOAD DATA -----
# Place your Kaggle CSVs in data/raw/ and update these paths
SLAMS_PATH = '../data/raw/grand_slam_winners.csv'
ATP_PATH = '../data/raw/atp_matches_2000_2025.csv'

slams = pd.read_csv(SLAMS_PATH)
atp = pd.read_csv(ATP_PATH)

print('Slams shape:', slams.shape)
print('ATP shape:', atp.shape)
slams.head()

KeyboardInterrupt: 

In [ ]:
# ----- COLUMN INSPECTION -----
# IMPORTANT: Column names vary by Kaggle dataset version.
# Run this cell first and map to the canonical names below.
print('SLAM columns:', slams.columns.tolist())
print('ATP columns:', atp.columns.tolist())

In [ ]:
# ----- COLUMN RENAMING (adjust as needed) -----
# Example mapping — update after inspecting above
slams = slams.rename(columns={
    'Winner': 'winner',       # player who won the Slam
    'Year': 'year',
    'Tournament': 'tournament',
})

atp = atp.rename(columns={
    'winner_name': 'winner',
    'loser_name': 'loser',
    'tourney_date': 'date',
    'surface': 'surface',
    'tourney_level': 'level',
})

In [ ]:
# ----- GRAND SLAM WINS BY PLAYER (all time) -----
top_winners = slams['winner'].value_counts().head(20)

fig, ax = plt.subplots()
top_winners.plot(kind='bar', ax=ax, color='steelblue')
ax.set_title('All-Time Grand Slam Wins — Top 20 Players')
ax.set_xlabel('Player')
ax.set_ylabel('Slams Won')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.savefig('../outputs/slam_wins_alltime.png', dpi=150)
plt.show()

In [ ]:
# ----- DECADE-BY-DECADE CONCENTRATION -----
slams['decade'] = (slams['year'] // 10) * 10

# HHI per decade (from our stats module)
import sys; sys.path.insert(0, '..')
from src.stats.significance import herfindahl_hirschman_index

hhi_by_decade = slams.groupby('decade')['winner'].apply(herfindahl_hirschman_index)
print(hhi_by_decade)

hhi_by_decade.plot(kind='bar', color='coral', title='Slam Title Concentration (HHI) by Decade')
plt.ylabel('HHI (0=even, 1=monopoly)')
plt.tight_layout()
plt.savefig('../outputs/hhi_by_decade.png', dpi=150)
plt.show()

In [ ]:
# ----- ATP DATA: DATE RANGE & SURFACE SPLIT -----
atp['date'] = pd.to_datetime(atp['date'].astype(str), format='%Y%m%d', errors='coerce')
print(f"Date range: {atp['date'].min()} → {atp['date'].max()}")
print(f"Missing dates: {atp['date'].isna().sum()}")
print()
print('Surface distribution:')
print(atp['surface'].value_counts())

In [ ]:
# ----- SAVE CLEANED DATA -----
slams.to_csv('../data/processed/slams_clean.csv', index=False)
atp.to_csv('../data/processed/atp_clean.csv', index=False)
print('Saved to data/processed/')